In [2]:
import csv
import json
import requests

In [3]:
baseURL = 'https://www.loc.gov'
endpoint = '/free-to-use'
collection = '/libraries'
parameters ={
    'fo':'json'
}

In [4]:
collection_list_response = requests.get(baseURL + endpoint + collection, params=parameters)

print(collection_list_response.url)

https://www.loc.gov/free-to-use/libraries?fo=json


In [5]:
collection_list_response.status_code

200

In [6]:
collection_json = collection_list_response.json()

print(json.dumps(collection_json, indent=2))

{
  "breadcrumbs": [
    {
      "Library of Congress": "https://www.loc.gov"
    },
    {
      "Library of Congress Free to Use and Reuse": "https://www.loc.gov/free-to-use/"
    },
    {
      "Libraries": "https://www.loc.gov/free-to-use/libraries/"
    }
  ],
  "content": {
    "active": true,
    "components": [
      {
        "blurb": "For greater knowledge, on more subjects, use your library more often. Illinois WPA Arts Project, 1936-1941. Prints & Photographs Division, Noyes Library for Young Children. Kensington, Maryland. Photo by Carol M. Highsmith,  2011. Prints & Photographs Division, Bethune-Cookman College. Students in the library reading room, Daytona Beach, Florida. Gordon Parks, 1943. Prints & Photographs Division, Public library in Antonito,  Colorado, near the New Mexico border. Photo by Carol M. Highsmith, 2015. Prints & Photographs Division, Library and Innovation Center in New Castle, Delaware, defines itself as \"a community learning and resource hub.\" Photo

In [8]:
collection_json.keys()
for k in collection_json['content']['components'][0]['masonry_gallery'][0]['items']:
    print(k)

{'image': '/static/portals/free-to-use/public-domain/libraries/libraries-1.jpg', 'link': '/resource/cph.3f05183/', 'title': 'For greater knowledge, on more subjects, use your library more often. Illinois WPA Arts Project, 1936-1941. Prints & Photographs Division'}
{'image': '/static/portals/free-to-use/public-domain/libraries/libraries-2.jpg', 'link': '/resource/highsm.20336/', 'title': 'Noyes Library for Young Children. Kensington, Maryland. Photo by Carol M. Highsmith,  2011. Prints & Photographs Division'}
{'image': '/static/portals/free-to-use/public-domain/libraries/libraries-3.jpg', 'link': '/resource/fsa.8d24709/', 'title': 'Bethune-Cookman College. Students in the library reading room, Daytona Beach, Florida. Gordon Parks, 1943. Prints & Photographs Division'}
{'image': '/static/portals/free-to-use/public-domain/libraries/libraries-4.jpg', 'link': '/resource/highsm.36052/', 'title': 'Public library in Antonito,  Colorado, near the New Mexico border. Photo by Carol M. Highsmith,

In [11]:
collection_set_list = 'outputs/libraries_list.csv'
headers = ['image','link','title']

with open(collection_set_list, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    for item in collection_json['content']['components'][0]['masonry_gallery'][0]['items']:
        item['title'] = item['title'].rstrip()
        writer.writerow(item)
    print('wrote',collection_set_list)

wrote outputs/libraries_list.csv


In [12]:
import json

with open('outputs/libraries_set.json', 'w', encoding='utf-8') as f:
  json.dump(collection_json, f, indent=2)

print('Saved raw collection JSON!')


Saved raw collection JSON!


In [13]:
len(collection_json['content']['components'][0]['masonry_gallery'][0]['items'])

62

In [ ]:
import glob
from pathlib import Path

In [15]:
def regenerate_collection_list(collection_csv):

    coll_items = list()

    with open(collection_csv, 'r', newline='', encoding='utf-8') as f:
        data = csv.DictReader(f)

        for row in data:
            row_dict = dict()
            for field in data.fieldnames:
                row_dict[field] = row[field]
            coll_items.append(row_dict)

        return coll_items

In [17]:
collection_csv = Path('outputs', 'libraries_list.csv')
collection_set_list = regenerate_collection_list(collection_csv)
collection_set_list[0]

{'image': '/static/portals/free-to-use/public-domain/libraries/libraries-1.jpg',
 'link': '/resource/cph.3f05183/',
 'title': 'For greater knowledge, on more subjects, use your library more often. Illinois WPA Arts Project, 1936-1941. Prints & Photographs Division'}

In [18]:
baseURL = 'https://loc.gov'
parameters = {'fo': 'json'}

item_metadata_directory = Path('outputs', 'item-metadata').mkdir(
    parents=True, exist_ok=True
)


In [29]:
import os

item_count = 0
error_count = 0
file_count = 0

data_directory = 'outputs'
item_metadata_directory = 'item-metadata'
item_metadata_file_prefix = 'item_metadata'
json_suffix = '.json'

for item in collection_set_list:
  if item['link'] == 'link':
    continue
  # these resource links could redirect to item pages, but currently don't work
  if '?' in item['link']:
    resource_ID = item['link']
    short_ID = item['link'].split('/')[2]
    item_metadata = requests.get(
        baseURL + resource_ID, params={'fo': 'json'}
    )
    print('requested', item_metadata.url, item_metadata.status_code)
    if item_metadata.status_code != 200:
      print('requested', item_metadata.url, item_metadata.status_code)
      error_count += 1
      continue
    try:
      item_metadata.json()
    except:  # basically this catches all of the highsmith photos with hhh in the ID
      error_count += 1
      print('no json found')
      continue
    fout = os.path.join(
        data_directory,
        item_metadata_directory,
        str(item_metadata_file_prefix + '-' + short_ID + json_suffix),
    )
    with open(fout, 'w', encoding='utf-8') as json_file:
      json_file.write(json.dumps(item_metadata.json()['item']))
      file_count += 1
      print('wrote', fout)
    item_count += 1
  else:
    resource_ID = item['link']
    short_ID = item['link'].split('/')[2]
    item_metadata = requests.get(
        baseURL + resource_ID, params={'fo': 'json'}
    )
    print('requested', item_metadata.url, item_metadata.status_code)
    if item_metadata.status_code != 200:
      print('requested', item_metadata.url, item_metadata.status_code)
      error_count += 1
      continue
    try:
      item_metadata.json()
    except:
      error_count += 1
      print('no json found')
      continue
    fout = os.path.join(
        data_directory,
        item_metadata_directory,
        str(item_metadata_file_prefix + '-' + short_ID + json_suffix),
    )
    with open(fout, 'w', encoding='utf-8') as json_file:
      json_file.write(json.dumps(item_metadata.json()['item']))
      file_count += 1
      print('wrote', fout)
    item_count += 1

print('--- mini LOG ---')
print('items requested:', item_count)
print('errors:', error_count)
print('files written:', file_count)


requested https://www.loc.gov/resource/hhh.dc0121.photos/?fo=json&sp=24 200
wrote outputs\item-metadata\item_metadata-hhh.dc0121.photos.json
requested https://www.loc.gov/resource/ppbd.00600/?fo=json 200
wrote outputs\item-metadata\item_metadata-ppbd.00600.json
requested https://www.loc.gov/resource/fsa.8b14169/?fo=json 200
wrote outputs\item-metadata\item_metadata-fsa.8b14169.json
requested https://www.loc.gov/resource/highsm.32720/?fo=json 200
wrote outputs\item-metadata\item_metadata-highsm.32720.json
requested https://www.loc.gov/resource/fsa.8c22565/?fo=json 200
wrote outputs\item-metadata\item_metadata-fsa.8c22565.json
requested https://www.loc.gov/resource/mrg.00785/?fo=json 200
wrote outputs\item-metadata\item_metadata-mrg.00785.json
requested https://www.loc.gov/resource/hhh.ak0345.photos/?fo=json&sp=2 200
wrote outputs\item-metadata\item_metadata-hhh.ak0345.photos.json
requested https://www.loc.gov/resource/highsm.43863/?fo=json 200
wrote outputs\item-metadata\item_metadata-h

In [30]:
data_dir = 'outputs'
files_dir = 'item-files'
metadata_dir = 'item-metadata'

files_loc = Path(data_dir, files_dir).mkdir(exist_ok=True)

In [31]:
search_for_metadata_here = Path(data_dir, metadata_dir)
print(search_for_metadata_here)

metadata_file_list = list(search_for_metadata_here.glob('*.json'))
print(len(metadata_file_list))
print(metadata_file_list)

outputs\item-metadata
59
[WindowsPath('outputs/item-metadata/item_metadata-cph.3b41963.json'), WindowsPath('outputs/item-metadata/item_metadata-cph.3b43255.json'), WindowsPath('outputs/item-metadata/item_metadata-cph.3c18157.json'), WindowsPath('outputs/item-metadata/item_metadata-cph.3f05168.json'), WindowsPath('outputs/item-metadata/item_metadata-cph.3f05183.json'), WindowsPath('outputs/item-metadata/item_metadata-det.4a17925.json'), WindowsPath('outputs/item-metadata/item_metadata-det.4a23603.json'), WindowsPath('outputs/item-metadata/item_metadata-ds.06507.json'), WindowsPath('outputs/item-metadata/item_metadata-ds.06560.json'), WindowsPath('outputs/item-metadata/item_metadata-fsa.8b14169.json'), WindowsPath('outputs/item-metadata/item_metadata-fsa.8b32222.json'), WindowsPath('outputs/item-metadata/item_metadata-fsa.8c22565.json'), WindowsPath('outputs/item-metadata/item_metadata-fsa.8d24709.json'), WindowsPath('outputs/item-metadata/item_metadata-g3851e.ct006252.json'), WindowsPat

In [32]:
item_image_urls = list()
count = 0

for item in metadata_file_list:
  with open(item, 'r', encoding='utf-8') as f:
    metadata = json.load(f)
    image_url_no = len(metadata['image_url'])
    image_url = metadata['image_url']
    item_image_urls.append(image_url)
    count += 1

print(f'Identified { str(count) } image URLs')

Identified 59 image URLs


In [35]:
collection_set_list_with_images = list()

for item in metadata_file_list:
  with open(item, 'r', encoding='utf-8') as item_info:
    item_metadata = json.load(item_info)

  item_metadata_dict = dict()
  item_metadata_dict['item_URI'] = item_metadata['id']
  try:
    item_metadata_dict['lccn'] = item_metadata['library_of_congress_control_number']
  except:
    item_metadata_dict['lccn'] = None
  item_metadata_dict['title'] = item_metadata['title']
  item_metadata_dict['image_URL_large'] = item_metadata['image_url'][-1]

  collection_set_list_with_images.append(item_metadata_dict)

print(collection_set_list_with_images)


[{'item_URI': 'http://www.loc.gov/item/91787443/', 'lccn': '91787443', 'title': 'Carnegie Library, Cordele, Georgia', 'image_URL_large': 'https://tile.loc.gov/storage-services/service/pnp/cph/3b40000/3b41000/3b41900/3b41963v.jpg#h=823&w=1024'}, {'item_URI': 'http://www.loc.gov/item/89710983/', 'lccn': '89710983', 'title': 'Silence / Mabel Dwight.', 'image_URL_large': 'https://tile.loc.gov/storage-services/service/pnp/cph/3b40000/3b43000/3b43200/3b43255r.jpg#h=422&w=640'}, {'item_URI': 'http://www.loc.gov/item/97511671/', 'lccn': '97511671', 'title': 'Carnegie Library, Sheldon, Iowa', 'image_URL_large': 'https://tile.loc.gov/storage-services/service/pnp/cph/3c10000/3c18000/3c18100/3c18157v.jpg#h=824&w=1024'}, {'item_URI': 'http://www.loc.gov/item/98508385/', 'lccn': '98508385', 'title': 'Curb service 10,000 current books - convenient, free, time saving : Chicago Public Library, Randolph St. corridor.', 'image_URL_large': 'https://tile.loc.gov/storage-services/service/pnp/cph/3f00000/3f0

In [37]:
item_count = 0
error_count = 0
file_count = 0

img_file_prefix = 'img_'

for item in collection_set_list_with_images:
  image_URL = item['image_URL_large']
  short_ID = item['item_URI'].split('/')[-2]
  print('... requesting', image_URL)
  item_count += 1

  # if found, save image
  r = requests.get(image_URL)
  if r.status_code == 200:
    img_out = os.path.join(
        data_dir, files_dir, str(img_file_prefix + short_ID + '.jpg')
    )
    with open(img_out, 'wb') as file:
      file.write(r.content)
      print('Saved', img_out)
      file_count += 1
  else:
    error_count += 1

print('--- mini LOG ---')
print('files requested:', item_count)
print('errors:', error_count)
print('files written:', file_count)


... requesting https://tile.loc.gov/storage-services/service/pnp/cph/3b40000/3b41000/3b41900/3b41963v.jpg#h=823&w=1024
Saved outputs\item-files\img_91787443.jpg
... requesting https://tile.loc.gov/storage-services/service/pnp/cph/3b40000/3b43000/3b43200/3b43255r.jpg#h=422&w=640
Saved outputs\item-files\img_89710983.jpg
... requesting https://tile.loc.gov/storage-services/service/pnp/cph/3c10000/3c18000/3c18100/3c18157v.jpg#h=824&w=1024
Saved outputs\item-files\img_97511671.jpg
... requesting https://tile.loc.gov/storage-services/service/pnp/cph/3f00000/3f05000/3f05100/3f05168v.jpg#h=1024&w=708
Saved outputs\item-files\img_98508385.jpg
... requesting https://tile.loc.gov/storage-services/service/pnp/cph/3f00000/3f05000/3f05100/3f05183v.jpg#h=1024&w=705
Saved outputs\item-files\img_98508155.jpg
... requesting https://tile.loc.gov/storage-services/service/pnp/det/4a10000/4a17000/4a17900/4a17925v.jpg#h=814&w=1024
Saved outputs\item-files\img_2016809661.jpg
... requesting https://tile.loc.g